# CSI2PointCloud — entrenamiento, métricas y visualización

Notebook del pipeline con el **dataset propio** (ESP32 + Kinect, 6 clases) en formato MM-Fi.
Usa el **modelo de Määttä** y el **dataset** incluidos en este repo.

- Split **70/15/15 cronológico por acción** (sin fuga temporal).
- Métricas **ICP** (fitness, RMSE, Chamfer) sobre el test reservado.
- Visualización GT vs predicho con los **colores de Määttä** (viridis por distancia).

> Ejecuta las celdas en orden (Run All). En CPU el entrenamiento tarda; con GPU es mucho más rápido.

## 1. Configuración y rutas

In [ ]:
import os, sys, glob, time, json
import numpy as np, scipy.io as scio, torch
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

# rutas: por defecto el modelo y el dataset INCLUIDOS en el repo (carpeta padre de /pc)
HERE = os.path.abspath('')
REPO = os.environ.get('CSI2PC_REPO', os.path.join(HERE, '..', 'csi2pointcloud'))
DATA = os.environ.get('CSI2PC_DATA', os.path.join(HERE, '..', 'mmfi_dataset'))
OUT  = os.environ.get('CSI2PC_OUT',  os.path.join(HERE, 'resultados'))
os.makedirs(os.path.join(OUT, 'modelo'), exist_ok=True)
sys.path.append(REPO)
from src.csi2pointcloud.csi2pointcloud_arch01 import CSI2PointCloudModel
from src.loss.mmslab_loss import CSI2PointCloudLoss

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Modelo:', REPO)
print('Datos :', DATA)
print('Device:', device)

## 2. Dataset y split 70/15/15 cronológico por acción

In [ ]:
from torch.utils.data import Dataset

class CSIDataset(Dataset):
    def __init__(self, root):
        self.items, self.actions = [], []
        for mat in sorted(glob.glob(os.path.join(root, '**', 'wifi-csi', '*.mat'), recursive=True)):
            b = mat.replace('wifi-csi', 'lidar').replace('.mat', '.bin')
            if os.path.exists(b):
                self.items.append((mat, b))
                self.actions.append(os.path.basename(os.path.dirname(os.path.dirname(mat))))
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        mat, binf = self.items[i]
        m = scio.loadmat(mat)
        amp = m['CSIamp'].astype(np.float64); pha = m['CSIphase'].astype(np.float64)
        amp[np.isinf(amp)] = np.nan; pha[np.isinf(pha)] = np.nan
        if np.any(np.isnan(amp)): amp[np.isnan(amp)] = np.nanmean(amp)
        if np.any(np.isnan(pha)): pha[np.isnan(pha)] = np.nanmean(pha)
        amp = (amp - np.nanmin(amp)) / (np.nanmax(amp) - np.nanmin(amp) + 1e-9)
        csi = np.stack([amp, pha], axis=2).astype(np.float32)
        x = torch.from_numpy(csi); x = (x - x.mean()) / (x.std() + 1e-9)
        gt = np.frombuffer(open(binf, 'rb').read(), dtype=np.float64).reshape(-1, 3).astype(np.float32)
        g = torch.from_numpy(gt.copy()); g = (g - g.mean()) / (g.std() + 1e-9)
        return x, g

def split_por_accion(ds, ratios=(0.70, 0.15, 0.15)):
    tr, va, te = [], [], []
    acts = np.array(ds.actions)
    for a in sorted(set(ds.actions)):
        idx = np.where(acts == a)[0]
        n = len(idx); n_tr = int(ratios[0]*n); n_va = int(ratios[1]*n)
        tr += idx[:n_tr].tolist(); va += idx[n_tr:n_tr+n_va].tolist(); te += idx[n_tr+n_va:].tolist()
    return tr, va, te

ds = CSIDataset(DATA)
tr, va, te = split_por_accion(ds)
json.dump({'train': tr, 'val': va, 'test': te}, open(os.path.join(OUT, 'split.json'), 'w'))
print('Frames:', len(ds), '| train', len(tr), '| val', len(va), '| test', len(te))
print('Acciones:', sorted(set(ds.actions)))

## 3. Entrenamiento
Ajusta `EPOCHS`. Guarda el mejor modelo por validación en `resultados/modelo/csi2pc_v2.pth`.

In [ ]:
EPOCHS = 60
BATCH  = 8
torch.manual_seed(0)

train_dl = DataLoader(Subset(ds, tr), batch_size=BATCH, shuffle=True)
val_dl   = DataLoader(Subset(ds, va), batch_size=BATCH)

model = CSI2PointCloudModel(256, 4, 3, 3, 1200, 3, 114, 10).to(device)
loss_fn = CSI2PointCloudLoss()
opt = torch.optim.NAdam(model.parameters(), lr=1e-4)

hist_tr, hist_val, best = [], [], None
for ep in range(EPOCHS):
    t0 = time.time(); model.train(); tl = []
    for x, g in train_dl:
        x, g = x.to(device), g.to(device)
        opt.zero_grad(); pred, tf = model(x); loss = loss_fn(pred, g, tf)
        loss.backward(); opt.step(); tl.append(loss.item())
    model.eval(); vl = []
    with torch.no_grad():
        for x, g in val_dl:
            x, g = x.to(device), g.to(device)
            pred, tf = model(x); vl.append(loss_fn(pred, g, tf).item())
    hist_tr.append(np.mean(tl)); hist_val.append(np.mean(vl) if vl else np.nan)
    print('Epoca %3d/%d | train %.1f | val %.1f | %.1fs' % (ep+1, EPOCHS, hist_tr[-1], hist_val[-1], time.time()-t0))
    if best is None or hist_val[-1] < best:
        best = hist_val[-1]
        torch.save({'model_state_dict': model.state_dict(), 'epochs': ep+1}, os.path.join(OUT, 'modelo', 'csi2pc_v2.pth'))
print('LISTO. Mejor val =', round(best, 2))

## 4. Curva de pérdida

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(hist_tr, label='train'); plt.plot(hist_val, label='val')
plt.xlabel('epoca'); plt.ylabel('perdida (Chamfer + reg)'); plt.legend(); plt.grid(True)
plt.title('Entrenamiento CSI2PointCloud (6 clases)')
plt.savefig(os.path.join(OUT, 'curva_perdida.png'), dpi=120, bbox_inches='tight')
plt.show()

## 5. Métricas ICP sobre el TEST (fitness, RMSE, Chamfer)

In [ ]:
import open3d as o3d
THR = 0.10  # umbral de correspondencia en metros

ckpt = torch.load(os.path.join(OUT, 'modelo', 'csi2pc_v2.pth'), map_location=device)
model.load_state_dict(ckpt['model_state_dict']); model.eval()

def nn_dist(A, B):
    kdt = o3d.geometry.KDTreeFlann(o3d.geometry.PointCloud(o3d.utility.Vector3dVector(B)))
    d = np.empty(len(A))
    for i, p in enumerate(A):
        _, _, d2 = kdt.search_knn_vector_3d(p, 1); d[i] = np.sqrt(d2[0])
    return d

fits, rmses, chamfs = [], [], []
for i in te:
    gt = np.frombuffer(open(ds.items[i][1], 'rb').read(), dtype=np.float64).reshape(-1,3)
    x, _ = ds[i]
    with torch.no_grad():
        pred, _ = model(x.unsqueeze(0).to(device))
    pred = pred.squeeze(0).cpu().numpy().astype(np.float64)
    pred_m = pred * gt.std() + gt.mean()
    reg = o3d.pipelines.registration.registration_icp(
        o3d.geometry.PointCloud(o3d.utility.Vector3dVector(pred_m)),
        o3d.geometry.PointCloud(o3d.utility.Vector3dVector(gt)),
        THR, np.eye(4),
        o3d.pipelines.registration.TransformationEstimationPointToPoint())
    fits.append(reg.fitness); rmses.append(reg.inlier_rmse)
    chamfs.append(nn_dist(pred_m, gt).mean() + nn_dist(gt, pred_m).mean())

print('TEST (%d frames, umbral %.2f m)' % (len(fits), THR))
print('  ICP fitness : %.4f' % np.mean(fits))
print('  ICP RMSE    : %.4f m' % np.mean(rmses))
print('  Chamfer     : %.4f m' % np.mean(chamfs))

## 6. Visualización GT vs Predicho (colores de Määttä)

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa

def plot_maata(ax, P, titulo):
    h, prof, alt = P[:,0], P[:,2], -P[:,1]
    dist = np.linalg.norm(P, axis=1)
    ax.scatter(h, prof, alt, c=dist, cmap='viridis', s=7, alpha=0.9)
    ax.set_title(titulo); ax.set_xlabel('X'); ax.set_ylabel('Z'); ax.set_zlabel('altura (-Y)')
    ax.view_init(elev=12, azim=-75)

i = te[len(te)//2]  # un frame del test
x, g = ds[i]
with torch.no_grad():
    pred, _ = model(x.unsqueeze(0).to(device))
pred = pred.squeeze(0).cpu().numpy(); gt = g.numpy()

fig = plt.figure(figsize=(13,6))
plot_maata(fig.add_subplot(1,2,1, projection='3d'), gt,   'Ground-truth (Kinect)  [%s]' % ds.actions[i])
plot_maata(fig.add_subplot(1,2,2, projection='3d'), pred, 'Predicho (CSI -> IA)')
plt.suptitle('Generacion de nube de puntos - colores de Maata (viridis por distancia)')
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'pred_vs_gt.png'), dpi=120, bbox_inches='tight')
plt.show()